# Regular Expressions

In this chapter, we examine a powerful tool for manipulating strings, regular expressions, often abbreviated as regex. Regular expressions are a way to describe a set of strings by using a single string to represent them. They are called "regular" as they have (at least) the expressive power as a regular grammar (for a more in-depth discussion of the hierarchy of formal languages, see Part 1).

In practice, while regular expressions do describe sets of strings, they can be used for a multitude of goals, among which:
- Check whether a string matches a given expression
- Find one or more occurrences of a regex in a string
- Extract information from a string
- Find one or more occurrences of a regex and replace them with another string
- Split a string into multiple ones

## Regular Expressions in Python

In Python, the implementation of regular expressions is found in a package called `re`, which contains multiple functions related to regular expressions. This package is part of the standard library of Python, meaning that it is included in any installation. In order to use this package, it must be imported using the `import` keyword.


In [2]:
import re

In this section examine the most important functionalities of the `re` package, while the rest can be found in the official documentation at <https://docs.python.org/3/library/re.html>.

The simplest form of a regular expression is a string composed of characters, digits, spaces. If none of the reserved characters are used (such as `.[]{}()+*`) this is treated as the regular expression that will match only the specified string. These can be used to understand the behaviour of the different functions of the `re` package.

## re.match
`re.match(pattern, string)` looks for `pattern` at the beginning of `string`. it returns a `Match` object if a match was found, `None` otherwise.

The following examples show its behaviour:

In [3]:
simple_regex = "my"
test_string = "my first regex"
print(re.match(simple_regex, test_string))

<re.Match object; span=(0, 2), match='my'>


Notice that in this example `re.match` returns a `Match` object that contains the match "my", since it appears at the beginning of `test_string`. The `span` attribute shown in the ouptut contains the indices spanning the match inside of the string. In this case, this means that the match can be found from the character at index 0 to the character at index 2 (excluded) in the original string.

A more in-depth discussion of the `Match` object can be found in [a later section](#the-match-object).

In [4]:
simple_regex = "first"
test_string = "my first regex"
print(re.match(simple_regex, test_string))

None


In this example, "first" does not appear at the beginning of `test_string`. Therefore, the return value of `match` is `None` as shown in the output.


## re.search

`re.search(pattern, string)` searches for the first match of `pattern` in `string`. The match can be anywhere in `string`, not just at the beginning. It returns a `Match` object if a match was found, `None` otherwise.

The following examples show its behaviour:

In [5]:
simple_regex = "first"
test_string = "my first regex"
print(re.search(simple_regex, test_string))

<re.Match object; span=(3, 8), match='first'>


In this example, the pattern "first" can be found in `test_string` between indices 3 and 8 (excluded).

In [6]:
simple_regex = "First"
test_string = "my first regex"
print(re.search(simple_regex, test_string))

None


In this second example, notice that regular expressions are case sensitive, so the pattern "First" is not present in `test_string`. Therefore, `re.search` returns `None`.

## re.findall

`re.findall(pattern, string)` finds all the matches of `pattern` in `string`. It returns a list of strings (or a list of tuples containing strings if groups are present in the regex). If no match can be found, the function returns an empty list.

Here are some examples of its behavior:

In [7]:
simple_regex = "an"
test_string = "an apple, an umbrella"
print(re.findall(simple_regex, test_string))

['an', 'an']


In the previous code, `re.findall` returns a list of strings, one for each match of "an" that was found.

In [8]:
simple_regex = "an"
test_string = "an apple and an orange"
print(re.findall(simple_regex, test_string))

['an', 'an', 'an', 'an']


Notice that in the previous example four matches are found, even if the word "an" appears only twice inside `test_string`. This is because, in addition to the two occurrences of the word "an", the string "an" is found also in "<u>an</u>d" and "or<u>an</u>ge". This is due to the fact that regular expressions simply look for any match of the specified pattern, even inside words.

In [9]:
simple_regex = "banana"
test_string = "an apple and an orange"
print(re.findall(simple_regex, test_string))

[]


As shown above, if no match is found `findall` returns an empty list.

## re.finditer
`re.finditer(pattern,string)` it finds all the matches of `pattern` in `string`. It returns an iterator of `Match` objects.

This function, at first glance, is very similar to `re.findall`. However, the function returns an iterator, an object representing a stream of data that can be iterated over and that yields one `Match` object at a time, until no more can be found and the `StopIteration` exception is raised.

In order to understand its behaviour, let's see some examples:

In [10]:
simple_regex = "an"
test_string = "an apple, an umbrella"
print(re.finditer(simple_regex, test_string))

As you can see, if we print the output of `finditer` we only learn that it returns a `callable_iterator` object, but we do not get any information about its contents. In order to obtain the match objects, we need to use the `next(iterator)` function to obtain each `Match`

In [ ]:
simple_regex = "an"
test_string = "an apple, an umbrella"
matches_iterator = re.finditer(simple_regex, test_string)

print("first match")
print(next(matches_iterator)) # first match

print("second match")
print(next(matches_iterator)) # second match

print("there is no third match, this raises a StopIteration exception")
print(next(matches_iterator)) # third match?

first match
<re.Match object; span=(0, 2), match='an'>
second match
<re.Match object; span=(10, 12), match='an'>
there is no third match, this raises a StopIteration exception


StopIteration: 

We can iteratively obtain each match by using the built-in `next(iterator)` function. What we obtain is a series of `Match` object that represent each match of `pattern` in `string`. When no more matches can be found, a `StopIteration` exception is raised.

A simpler alternative to access the matches is to use a `for` loop to iterate over them:

In [ ]:
simple_regex = "an"
test_string = "an apple, an umbrella"
matches_iterator = re.finditer(simple_regex, test_string)

for match in matches_iterator:
    print(match)

<re.Match object; span=(0, 2), match='an'>
<re.Match object; span=(10, 12), match='an'>


Finally, we can also construct a list from the iterator as follows:

In [ ]:
simple_regex = "an"
test_string = "an apple, an umbrella"
matches_iterator = re.finditer(simple_regex, test_string)
matches_list = list(matches_iterator)
print(matches_list)

[<re.Match object; span=(0, 2), match='an'>, <re.Match object; span=(10, 12), match='an'>]


## re.sub

`re.sub(pattern, replacement, string)` finds all the matches `pattern` in `string` and replaces them with `replacement`. It returns `string` with all the matches of `pattern` replaced.

Let's see some examples:

In [ ]:
simple_regex = "an"
replacement = "the"
test_string = "an apple, an umbrella"
print(re.sub(simple_regex, replacement, test_string))

the apple, the umbrella


In the previuos example, all instances of "an" get replaced by "the". The result is returned as a string by `re.sub`.

In [ ]:
simple_regex = "orange"
replacement = "the"
test_string = "an apple, an umbrella"
print(re.sub(simple_regex, replacement, test_string))

an apple, an umbrella


In the previous example, since no match for "orange" can be found, the unaltered `test_string` is returned by `re.sub`.

## re.split

`re.split(pattern, string)` it splits `string` whenever an occurrence of `pattern` is found and returns a list of strings resulting from the split.

Here are some examples of its application:

In [ ]:
test_string = "this-string-is-separated-by-dashes"
print(re.split("-", test_string))

['this', 'string', 'is', 'separated', 'by', 'dashes']


As shown in the previous example, the behaviour of `re.sub` is very similar to the `split` method implemented by strings in Python. The matches for the specified `pattern` are removed and the string is separated in two parts whenever a match is found.

In [ ]:
test_string = "this-string-is-separated-by-dashes"
print(re.split("_", test_string))

['this-string-is-separated-by-dashes']


If no match for `pattern` can be found like in the previous example, the function returns a list containing the entire, unaltered string.

## Quantifiers

Quantifiers are used to specify how many times the pattern to its left (a character, a character set or a group) can occurr. There are five types of quantifiers that can be used in Python regular expressions:
- `+` matches the previous pattern once or more than once;
- `*` matches the previous pattern zero or more times;
- `?` matches the previous pattern zero or one time;
- `{p}` where `p` is an integer matches the previous patter exactly `m` times
- `{p,q}`, where `p,q` are integers, matches the previous pattern between `p` and `q` times
- `{p,}`, where `p` is an integer, matches the previous pattern `p` or more times.

The following examples show the behaviour of all the quantifiers when applied to single characters, as we are yet to discuss character sets and groups.

In [ ]:
# match one or more a followed by b
print(re.findall("a+b", "aaab b"))

['aaab']


Notice that in the string "aaab", there are other matches for the regex "a+b" beside "aaab", as we could also match "ab" and "aab". However, regular expressions in Python are greedy, meaning that they will always return the longest possible sequence matching the specified pattern.

In [ ]:
# match a b that may or may not be preceded by one or more a
print(re.findall("a*b", "aaab b"))

['aaab', 'b']


In the prevous example, the matches for the "a*b" pattern are both "aaab" and "b", as "a\*" specifies that "a" can appear zero or more times, meaning that it is optional. 

In [ ]:
print(re.findall("a?b", "aaab b"))

['ab', 'b']


When using "a?b", the pattern indicates an optional "a" followed by a single "b". For this reason, the regex matches both "ab" from "aaab", as well as the single "b". 

In [ ]:
print(re.findall("a{2}b", "aaab b"))

['aab']


When specifying the quantity of "a" to match using `{2}`, the regular expression matches only "aab" from "aaab".

In [ ]:
print(re.findall("a{2,4}b", "aaab b ab"))

['aaab']


When a range is specified using `{2,4}`, the regular expression matches only `aaab` since it contains three "a", which are between 2 and 4. It does not, however, match "ab", as it only contains one a. In this case, like for "+" and "*", the greedy behaviour means that the highest number of "a"s possible is the match.

In [ ]:
print(re.findall("a{2,}b", "aaab b ab"))

Finally, we can specify a minimum of two "a" without a maximum as in the previous example.

## Character sets

Character sets are a functionality that is used to describe a set of characters that match in a given position of a regular expression. They are written as a set of characters inside squared brackets. For example, the regular expression `[ab]` matches a single character that can be either "a" or "b". Like mathematical (and Python) sets, the order of the characters is not relevant, meaning that `[ab]` is equivalent to `[ba]`. They can also be used in combination with quantifiers, meaning that quantifiers can be applied to the character set to its left.

In [22]:
print(re.findall("a[bcd]+", "ababd abb a"))

['ab', 'abd', 'abb']


In the previous example, the "a\[bcd\]+" pattern is used to indicate a sequence composed by an a followed by one or more "b", "c" or "d" characters.

When defining character sets, it is also possible to use "-" between two ASCII characters to define a character set using a range of ASCII characters. As an example, the expression "\[1-5\]\[0-9\]" matches strings of two digits, from 10 to 59. The same can be done for uppercase and lowercase letters, so "\[a-c\]\[m-n\]" matches a two letter sequence, where the first can be "a,b,c" and the second is either "m" or "n". To indicate any uppercase or lowercase letter "\[A-z\]" can be used. Finally, multiple ranges can be used in sequence the same character set, so "\[a-c0-3\]" matches a letter that is "a","b" or "c" or a digit that is "0", "1", "2" or "3"

In [23]:
print(re.findall("[a-c0-3]+", "a c2 h6"))

['a', 'c2']


In addition to defining a character set by indicating the characters that belong to it, the "^" character can also be used at the beginning of the group to indicate that the character set is formed by exclusion. As an example, "[^123]" is a character set that matches with every character, except "1", "2" and "3".

In [24]:
print(re.findall("0[^123]+", "01 0ab 0a1"))

['0ab 0a']


In the previous example, the regular expression matches the entire sequence "0ab 0a" since it is formed by a 0 followed by a series of characters that are neither "1", "2" or "3".

## Special Characters

Some characters have special roles in Python regular expressions: 
- `.` represents any character;
- `^` represents the beginning of a string;
- `$` represents the end of a string.

Let's see how they behave using a few examples:


In [16]:
print(re.findall(".n", "and enable"))

['an', 'en']


In the previous example, the pattern ".n" matches any character followed by n. Therefore, both "an" and "en" are matches. 

In [17]:
print(re.findall("^the", "the fox and the deer"))

['the']


When applying "^the" to the previous example, the pattern matches "the" only at the beginning of a string. Therefore, the only valid match is "the" at the beginning of the string.

In [18]:
print(re.findall("e$", "one more time"))

['e']


The previous expression "e$" matches "e" at the end of a string. In this example, the only valid match is "e" at the end of "time", while the "e" in "one" and "more" are not matches for the expression.

### Escaping characters and raw strings

So far we have discussed a series of special characters that have defined meanings in regular expressions, such as "\[", ".", "+". However, it is often useful to include these as simple characters in a regular expression. For example, we might be interested in matching a number with a fractional part, which would therefore contain a "." character. The naif regular expression "[0-9]+.[0-9]+" would not work, as it matches one or more digits followed by any single character, followed by one or more digits.

In [26]:
print(re.findall("[0-9]+.[0-9]+", "0.9 0b9"))

['0.9', '0b9']


In this example, the expression matches both "0.9" and "0b9" as they both contain a character between two digits. To solve this issue, we need to escape the dot using "\.". However, this is not possible when using regular strings, as "\\" is used to escape special characters, such as "\\n" for the new line character. For this reason, we use raw strings, which are obtained by prepending r before the definition of a string. By writing `r"\n"` for example, the sequence `\n` is not treated as a new line, instead it is treated as the sequence of characters "\n". An alternative would be to write `\\n`, escaping the "\\" character itself.

In [30]:
new_line = "\n"
escaped_string = "\\n"
raw_string = r"\n"

print(f"new line has length {len(new_line)}")
print(f"the escaped string has length {len(escaped_string)}")
print(f"the raw string has length {len(raw_string)}")

new line has length 1
the escaped string has length 2
the raw string has length 2


It is then possible to write an expression matching a number with a fractional part using a raw string as follows: 

In [31]:
print(re.findall(r"[0-9]+\.[0-9]+", "0.9 0b9"))

['0.9']


In general, it is often preferable to write all regular expressions as raw strings to avoid any issue.

## Special Sequences

In addition to special characters, some special sequences can be used to express character sets in a concise and effective way.
- `\w` matches any "word character" (letters, digits, "_", characters in any writing system)
- `\b` matches a "word boundary", meaning the place where the string transitions from a word character to a non-word character.
- `\d` indicates a unicode decimal digit, including both decimal digits as well as digits in other writing systems (for example, Chinese)
- `\s` indicates any Unicode spacing character (space, tab, etc).
These special sequences must be used in raw strings, or by escaping "\\" by writing "\\\\".

In addition to these special sequences, the opposite of each set is defined as the uppercase version of the sequence. So "\W" denotes non-word characters, "\S" denotes non spacing characters, and so on.

Notice that special sequences in Python fully support Unicode characters, meaning that they work even with non-ASCII characters and eve non Latin characters.

In [35]:
print(re.findall(r"\w+", "My Neighbor Totoro, it's my favourite"))
print(re.findall(r"\w+", "となりのトトロ"))

['My', 'Neighbor', 'Totoro', 'it', 's', 'my', 'favourite']
['となりのトトロ']


In the previous example, "\w+" denotes any sequence of word characters.Therefore, in the first example, we obtain a list of words in the sentence.
In the the second example, however, the string is not split, as it does not contain any spacing character.

The next example shows a regular expression containing "\\b".

In [38]:
print(re.findall(r"at", "bat, rat at"))
print(re.findall(r"\bat\b", "bat, rat at"))

['at', 'at', 'at']
['at']


As shown in the previous example, the word boundary can be used to find occurrences of a single word (in this case "at") in a string. In the first `print` statement, we simply use the pattern "at". This means that "at" is found three times, in "bat", "rat" and in "at", as we did not specify that "at" should be found within a word. In the second print statement, however, we use `r"\bat\b"` meaning that we are looking for "at" but it must be surrounded by word boundaries. This means that it must be surrounded by any non word character (including the beginning and end of the string).

## Groups

Groups are a sequence of characters, delimited by round parenthesis, that is used to isolate a portion of the pattern, to apply quantifiers to it, or to extract their content. They can be capturing or non capturing and in the case of capturing groups we are able to extract them and even use them in substitutions.

Let's see some examples of their behavior.

In [46]:
print(re.findall(r"(la)", "lllll lalala l a l a"))

['la', 'la', 'la']


In the previous example, the expression "(la)" matches a capturing group containing "la". When using "findall", all the matches are returned in a list.

In [48]:
print(re.findall(r"(\w+) (\w+) (\w+)", "a sunny day"))

[('a', 'sunny', 'day')]


In this case, since the pattern contains three groups, `findall` returns a list containing a tuple with one string for each group.

Capturing groups can be used to extract information that can be used in substitutions. In particular, the first group can be denoted in the replacement string as "\\1", the second as "\\2" and so on. For example, to replace the word "sunny" with "rainy" in the previous example, we would write:

In [ ]:
print(re.sub(r"(\w+) (\w+) (\w+)",r"\1 rainy \3", "a sunny day"))

a rainy day


One peculiarity has to do with the interaction of `findall`, groups and quantifiers. In particular, when using `findall` and a group that has a quantifier after it, the function will only return the last group.

In [ ]:
print(re.findall(r"(la)+", "lllll lalala l a l a"))

['la']

In order to avoid this issue, we can use non-capturing groups, which are just groups whose content is not captured.
They are denoted by writing "?:" at the beginning of the round parenthesis.

In [ ]:
print(re.findall(r"(?:la)+", "lllll lalala l a l a"))

['lalala']

Another possibility is to use `finditer`, as this function returns `Match` objects in a list, and each object contains the full match of the expression. 

In [55]:
matches_iterator = re.finditer(r"(la)+", "lllll lalala l a l a")
print(list(matches_iterator))

[<re.Match object; span=(6, 12), match='lalala'>]


## The Match object

The Match object is returned by `re.match` and `re.find`. It is also yielded by the iterator returned by `finditer`. So far, however, we have not discussed its behaviour, instead we just printed it. However, this object has useful methods that can be used to extract information from matches. In particular, by using the method `.group(idx)` we can extract information about the whole match or the strings extracted for each group. In particular, if the index is 0 we obtain the entire match, while 1, 2, 3 indicate the first, second third group and so on.

In [ ]:
date_str = "02/01/1999"
date_regex = r"([0-9]{2})/([0-9]{2})/([0-9]{4})"

match = re.search(date_regex,date_str)
print(match.group(0)) # the entire match
print(match.group(1)) # day
print(match.group(2)) # month
print(match.group(3)) # year

02/01/1999
02
01
1999


The same can be achieved by directly indexing the `Match` object:

In [ ]:
date_str = "02/01/1999"
date_regex = r"([0-9]{2})/([0-9]{2})/([0-9]{4})"
print(re.findall("[0-9]+.[0-9]+", "0.9 0b9"))
match = re.search(date_regex,date_str)
print(match[0]) # the entire date
print(match[1]) # day
print(match[2]) # month
print(match[3]) # year

02/01/1999
02
01
1999


When printing a `Match` object we observed that the output contains a `span=` portion, which indicates the beginning and the end of the full match of the regex. Unsurprisingly, this information can be accessed by using the `span(index)` method, which returns a tuple of indices indicating where the group begins and ends in the original string. Like for `.group(index)`, if we use 0 as index we obtain information about the whole match, while indices from 1 onwards indicate groups.

In [ ]:
date_str = "02/01/1999"
date_regex = r"([0-9]{2})/([0-9]{2})/([0-9]{4})"

match = re.search(date_regex,date_str)
print(match.span(0)) # the entire date
print(match.span(1)) # day

(0, 10)
(0, 2)


We can also use the methods `.start(index)` and `.end(index)` to obtain the starting and ending indices for each group. If 0 is used as index, the starting and ending indices for the whole stirng are provided, while values equal or lager than 1 produce the information about each group.

In [59]:
date_str = "02/01/1999"
date_regex = r"([0-9]{2})/([0-9]{2})/([0-9]{4})"

match = re.search(date_regex,date_str)
print(match.start(0)) # the entire date
print(match.end(1)) # day

0
2


## The "|" operator

The "|" operator is used to indicate an alternative between two or more expressions. As an example, the regex "one|of|these" matches either "one", "of" or "these" 

In [61]:
test_string = "one two three is used to count the number of things"
print(re.findall(r"one|of|these", test_string))

['one', 'of']


When using "|", the whole substring left of it is treated as one of the alternatives patterns for matching. For example:

In [62]:
test_string = "the beautiful coast"
test_regex = r"the beautiful mountain|coast"
print(re.search(test_regex, test_string))

<re.Match object; span=(14, 19), match='coast'>


In this case, the regex matches "the beautiful mountain" or "coast". Therefore, in our example, the only match is "coast". If we wanted to match both "the beautiful mountain" and "the beautiful coast", we could use "|" inside of a non-capturing group:

In [63]:
test_string = "the beautiful coast"
test_regex_group = r"the beautiful (?:mountain|coast)"
print(re.search(test_regex_group, test_string))

<re.Match object; span=(0, 19), match='the beautiful coast'>


In this case, the two alternative patterns for "|" are contained within the parenthesis.

## Compiling regular expressions

When using regular expressions, Python needs to read their content and compile them. While there is a mechanism that keeps in memory the most recently used patterns, it is also possible to compile the expressions that we need before execution. This is especially useful when we use multiple regular expressions in our code as it can speed up the execution of the program. To compile a regex, we use the `re.compile(pattern)` function, where `pattern` is a regular expression. This function returns a `re.Pattern` object that has `.match`, `.find`, `.findall`, `.finditer`, `.sub` and `.split` methods. These methods are equivalent to the respective functions, except for the fact that the first argument (`pattern`) is omitted.

In [64]:
fractional_number = re.compile(r"[0-9]+.[0-9]+")
print(fractional_number.findall("0.9 0b9"))

['0.9', '0b9']


## Additional Resources

- [Python documentation](https://docs.python.org/3/library/re.html): the official documentation of the `re` package
- [Regular Expresison HOWTO](https://docs.python.org/3/howto/regex.html): an introduction to regular expressions from the official Python documentation
- [regex101](https://regex101.com/): a website to test and visualize the behaviour of regular expressions